# Bronze Layer DDL — Table Definitions

## Overview
This notebook defines the Data Definition Language (DDL) for all **Bronze Layer** staging tables in Databricks Delta Lake.

The Bronze layer stores raw data exactly as received from the source systems (CRM and ERP), without transformations or quality filters.

### Staging Tables:
- **CRM Source System**:
  1. `bronze.crm_cust_info` — Customer demographics and contact info from CRM
  2. `bronze.crm_prd_info` — Product catalogue and pricing history from CRM
  3. `bronze.crm_sales_details` — Sales orders, quantities, and transactional lines from CRM
- **ERP Source System**:
  4. `bronze.erp_loc_a101` — Customer geographic locations from ERP
  5. `bronze.erp_cust_az12` — Customer birthdate and gender records from ERP
  6. `bronze.erp_px_cat_g1v2` — Product categories and maintenance classifications from ERP

All tables are created using the **Delta Lake** storage format (`USING DELTA`), providing ACID transactions, time travel, and schema enforcement.

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Helper display function fallback
try:
    display
except NameError:
    def display(df, n=20):
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure bronze database exists
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

## CRM Source Tables DDL
Define Bronze staging tables for CRM customer, product, and sales data.

In [ ]:
# COMMAND ----------
# 1. bronze.crm_cust_info
spark.sql("DROP TABLE IF EXISTS bronze.crm_cust_info")
spark.sql("""
CREATE TABLE bronze.crm_cust_info (
    cst_id INT,
    cst_key STRING,
    cst_firstname STRING,
    cst_lastname STRING,
    cst_marital_status STRING,
    cst_gndr STRING,
    cst_create_date DATE
)
USING DELTA
COMMENT 'Raw staging table for CRM customer information'
""")
print(">> Table created: bronze.crm_cust_info")

# 2. bronze.crm_prd_info
spark.sql("DROP TABLE IF EXISTS bronze.crm_prd_info")
spark.sql("""
CREATE TABLE bronze.crm_prd_info (
    prd_id INT,
    prd_key STRING,
    prd_nm STRING,
    prd_cost INT,
    prd_line STRING,
    prd_start_dt TIMESTAMP,
    prd_end_dt TIMESTAMP
)
USING DELTA
COMMENT 'Raw staging table for CRM product information'
""")
print(">> Table created: bronze.crm_prd_info")

# 3. bronze.crm_sales_details
spark.sql("DROP TABLE IF EXISTS bronze.crm_sales_details")
spark.sql("""
CREATE TABLE bronze.crm_sales_details (
    sls_ord_num STRING,
    sls_prd_key STRING,
    sls_cust_id INT,
    sls_order_dt INT,
    sls_ship_dt INT,
    sls_due_dt INT,
    sls_sales INT,
    sls_quantity INT,
    sls_price INT
)
USING DELTA
COMMENT 'Raw staging table for CRM sales order line details'
""")
print(">> Table created: bronze.crm_sales_details")

## ERP Source Tables DDL
Define Bronze staging tables for ERP customer locations, demographics, and product categories.

In [ ]:
# COMMAND ----------
# 4. bronze.erp_loc_a101
spark.sql("DROP TABLE IF EXISTS bronze.erp_loc_a101")
spark.sql("""
CREATE TABLE bronze.erp_loc_a101 (
    cid STRING,
    cntry STRING
)
USING DELTA
COMMENT 'Raw staging table for ERP customer geographic location mapping'
""")
print(">> Table created: bronze.erp_loc_a101")

# 5. bronze.erp_cust_az12
spark.sql("DROP TABLE IF EXISTS bronze.erp_cust_az12")
spark.sql("""
CREATE TABLE bronze.erp_cust_az12 (
    cid STRING,
    bdate DATE,
    gen STRING
)
USING DELTA
COMMENT 'Raw staging table for ERP customer birthdates and gender'
""")
print(">> Table created: bronze.erp_cust_az12")

# 6. bronze.erp_px_cat_g1v2
spark.sql("DROP TABLE IF EXISTS bronze.erp_px_cat_g1v2")
spark.sql("""
CREATE TABLE bronze.erp_px_cat_g1v2 (
    id STRING,
    cat STRING,
    subcat STRING,
    maintenance STRING
)
USING DELTA
COMMENT 'Raw staging table for ERP product category hierarchy and maintenance'
""")
print(">> Table created: bronze.erp_px_cat_g1v2")

## Table Verification
Confirm all 6 Bronze Delta tables are created in the metastore and display their schema metadata.

In [ ]:
# COMMAND ----------
# List all tables in the bronze database
df_bronze_tables = spark.sql("SHOW TABLES IN bronze")
display(df_bronze_tables)

# Inspect details of each bronze table
for table_row in df_bronze_tables.collect():
    t_name = f"bronze.{table_row['tableName']}"
    print(f"\n--- Schema Definition: {t_name} ---")
    spark.sql(f"DESCRIBE TABLE {t_name}").show(truncate=False)